# Grade & Event Raw Parsing Tester
This notebook allows you to test the raw DataFrame parsing logic from `src.services.ingestion` without any database setup. It purely shows how the data behaves when read from Excel or CSV.

In [ ]:
# Install dependencies if running in Colab (Uncomment if needed)
# !pip install pandas openpyxl ipywidgets

In [ ]:
import sys
import os

# Add the server project root to Python path so we can import 'src'
# If running via VSCode Jupyter in the 'server' folder, this adds '.'
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('..')) # fallback

import pandas as pd
import io

# Import only the raw ingestion utilities
from src.services.ingestion import (
    load_grades_dataframe, 
    load_attendance_dataframe, 
    _read_file, 
    detect_file_type
)

### 1. File Upload Field
Run this cell to display a file upload button. It will work both in Colab and local VS Code Jupyter.

In [ ]:
file_content = None
filename = ""

try:
    # Try Google Colab Upload first
    import google.colab.files
    print("Colab environment detected. Please upload a file:")
    uploaded = google.colab.files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        file_content = uploaded[filename]
        print(f"Loaded file: {filename}")
except ModuleNotFoundError:
    # Fallback to local ipywidgets upload for VS Code / Jupyter Lab
    import ipywidgets as widgets
    from IPython.display import display
    
    uploader = widgets.FileUpload(accept='.csv,.xlsx', multiple=False)
    print("Local environment detected. Please click the Upload button Below:")
    display(uploader)
    print("(After selecting a file, continue to the next cell to read it)")

### 2. Parse and Display DataFrame
Let's see how the raw DataFrame parser processes your file contents.

In [ ]:
# If using ipywidgets, we need to extract the content from the uploader in this step
if 'uploader' in locals() and uploader.value:
    if isinstance(uploader.value, dict): # ipywidgets < 8.0
        filename = list(uploader.value.keys())[0]
        file_content = uploader.value[filename]['content']
    else: # ipywidgets >= 8.0
        uploaded_file = uploader.value[0]
        filename = uploaded_file.name
        file_content = uploaded_file.content
    print(f"Extracted uploaded file: {filename}")

if file_content:
    content_type = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet' if filename.endswith('.xlsx') else 'text/csv'
    
    # 1. Read raw dataframe
    raw_df = _read_file(file_content, content_type)
    
    # 2. Detect file type based on column mappings
    file_type = detect_file_type(raw_df)
    print(f"Detected file type: {file_type}\n")
    
    # 3. Transform with domain logic
    if file_type == 'grades':
        parsed_df = load_grades_dataframe(raw_df)
    elif file_type == 'events':
        parsed_df = load_attendance_dataframe(raw_df)
    else:
        parsed_df = raw_df
        print("No specific parser found. Showing raw dataframe:\n")
        
    # Print some info and show the output
    print(f"Parsed shape (rows, columns): {parsed_df.shape}")
    display(parsed_df.head(20))
else:
    print("No file content loaded. Please upload a file in the previous cell.")